In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=PITbt8ypjQSMQDtyeK15BeOiHQMwNj&access_type=offline&code_challenge=RIpHUWSAaiO-FImy0pwdiwIrf-GKwPumwlReVkm-m60&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


Updates are available for some Google Clo

In [ ]:
!gcloud auth login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=UiDTZiS2COSkwLlSyy0l1pVKoF8Yfg&access_type=offline&code_challenge=4jMAWGLzSEe-ozTnSiVWYKCSv6hg9PyZzX727eiHbYk&code_challenge_method=S256


You are now logged in as [yt4@sanger.ac.uk].
Your current project is [open-targets-genetics-dev].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID


Updates are available for some Google Cloud CLI components.  To install them,
please run:
  $ gcloud components update



In [1]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment

"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize":"2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()

session= GentropySession()


path_to_release_folder="gs://open-targets-data-releases/25.03/"
#path_to_release_folder="gs://open-targets-pre-data-releases/24.12-uo_test-3/output/genetics/parquet/"
#path_to_release_folder="gs://ot_orchestration/releases/25.02_freeze1/"

si=StudyIndex.from_parquet(session,path_to_release_folder+"output/study/")
sl=StudyLocus.from_parquet(session,path_to_release_folder+"output/credible_set/")

Loading BokehJS ...

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/15 17:21:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/05/15 17:21:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


# EQTLs

In [2]:
eqtls=si.df.filter((f.col("studyType") == "eqtl")|(f.col("studyType") == "sceqtl"))

In [3]:
eqtls.show(1)

25/05/13 17:33:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+-----------+---------+--------------------+------------------------+----------+---------------+---------------------+-----------+--------+----------------+----------------------+---------------+------------------+----------------------------------+--------------------+-----------------+------+---------+--------+-------+---------------------+----------------+------------------+---------------+-------------+--------------------+-----------+---------+---------------+
|             studyId|  projectId|studyType|     traitFromSource|traitFromSourceMappedIds|diseaseIds|         geneId|biosampleFromSourceId|biosampleId|pubmedId|publicationTitle|publicationFirstAuthor|publicationDate|publicationJournal|backgroundTraitFromSourceMappedIds|backgroundDiseaseIds|initialSampleSize|nCases|nControls|nSamples|cohorts|ldPopulationStructure|discoverySamples|replicationSamples|qualityControls|analysisFlags|summarystatsLocation|hasSumstats|condition|sumstatQCValues|
+-----------------

In [4]:
eqtls_cs=sl.df.filter((f.col("studyType") == "eqtl")|(f.col("studyType") == "sceqtl")).cache()
eqtls_cs.show(1)

+--------------------+---------+---------------+----------+--------+--------------------+--------------------+--------+------+--------------+--------------+-------------------------------+-------------+-------------------+---------------+-----------------+----------------+------------------+------------+-----------+----------+--------+----------+-----+--------------------+--------------------+----------+
|        studyLocusId|studyType|      variantId|chromosome|position|              region|             studyId|    beta|zScore|pValueMantissa|pValueExponent|effectAlleleFrequencyFromSource|standardError|subStudyDescription|qualityControls|finemappingMethod|credibleSetIndex|credibleSetlog10BF|purityMeanR2|purityMinR2|locusStart|locusEnd|sampleSize|ldSet|               locus|          confidence|isTransQtl|
+--------------------+---------+---------------+----------+--------+--------------------+--------------------+--------+------+--------------+--------------+----------------------------

In [5]:
eqtls_cs=eqtls_cs.filter(f.col("isTransQtl")=="false").cache()

In [6]:
eqtls_cs.count()

1402446

In [7]:
combined_eqtls = eqtls_cs.join(
    eqtls, 
    on="studyId", 
    how="left"
).cache()
combined_eqtls.count()

1402446

In [8]:
combined_eqtls.show()

+--------------------+--------------------+---------+--------------------+----------+---------+--------------------+---------+------+--------------+--------------+-------------------------------+-------------+-------------------+---------------+-----------------+----------------+------------------+------------+-----------+----------+--------+----------+-----+--------------------+--------------------+----------+-----------+---------+--------------------+------------------------+----------+---------------+---------------------+-----------+--------+----------------+----------------------+---------------+------------------+----------------------------------+--------------------+-----------------+------+---------+--------+-------+---------------------+----------------+------------------+---------------+-------------+--------------------+-----------+--------------------+---------------+
|             studyId|        studyLocusId|studyType|           variantId|chromosome| position|           

In [9]:
combined_eqtls_s=combined_eqtls.select("geneId","biosampleId").distinct().cache()
combined_eqtls_s.count()

418887

In [ ]:
aggregated_eqtls = combined_eqtls.groupBy("geneId").agg(
    f.countDistinct("biosampleId").alias("distinct_biosample_count")
)
aggregated_eqtls.count()

ERROR:root:KeyboardInterrupt while sending command.             (128 + 8) / 200]
Traceback (most recent call last):
  File "/Users/yt4/Projects/gentropy/.venv/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/yt4/Projects/gentropy/.venv/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/yt4/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
##
target_constr=session.spark.read.parquet(path_to_release_folder+"output/target_prioritisation")

In [ ]:
target_constr.printSchema()

root
 |-- targetId: string (nullable = true)
 |-- isInMembrane: integer (nullable = true)
 |-- isSecreted: integer (nullable = true)
 |-- hasSafetyEvent: integer (nullable = true)
 |-- hasPocket: integer (nullable = true)
 |-- hasLigand: integer (nullable = true)
 |-- hasSmallMoleculeBinder: integer (nullable = true)
 |-- geneticConstraint: double (nullable = true)
 |-- paralogMaxIdentityPercentage: double (nullable = true)
 |-- mouseOrthologMaxIdentityPercentage: double (nullable = true)
 |-- isCancerDriverGene: integer (nullable = true)
 |-- hasTEP: integer (nullable = true)
 |-- mouseKOScore: double (nullable = true)
 |-- hasHighQualityChemicalProbes: integer (nullable = true)
 |-- maxClinicalTrialPhase: double (nullable = true)
 |-- tissueSpecificity: double (nullable = true)
 |-- tissueDistribution: double (nullable = true)



In [ ]:
##
target=session.spark.read.parquet(path_to_release_folder+"output/target/")

In [ ]:
target.printSchema()

root
 |-- id: string (nullable = true)
 |-- approvedSymbol: string (nullable = true)
 |-- biotype: string (nullable = true)
 |-- transcriptIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- canonicalTranscript: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: long (nullable = true)
 |    |-- end: long (nullable = true)
 |    |-- strand: string (nullable = true)
 |-- canonicalExons: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- genomicLocation: struct (nullable = true)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: long (nullable = true)
 |    |-- end: long (nullable = true)
 |    |-- strand: integer (nullable = true)
 |-- alternativeGenes: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- approvedName: string (nullable = true)
 |-- go: array (nullable = true)
 |    |-- element: struct (containsNull = tru

In [ ]:
target.show(1)

+---------------+--------------+--------------+--------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+---------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----+--------------------+--------------------+--------------+--------------------+--------------------+-----------------+--------------------+-------+
|             id|approvedSymbol|       biotype|       transcriptIds| canonicalTranscript|      canonicalExons|     genomicLocation|alternativeGenes|        approvedName|                  go|hallmarks|            synonyms|      symbolSynonyms|        nameSynonyms|functionDescriptions|subcellularLocations|         targetClass|     obsoleteSymbols|       obsoleteNames|          constraint| tep|          proteinIds|             dbXrefs|chemicalProbes|        

In [ ]:
target.groupBy("biotype").count().show()

+--------------------+-----+
|             biotype|count|
+--------------------+-----+
|transcribed_unita...|  181|
|     IG_C_pseudogene|    9|
|                rRNA|   47|
|              lncRNA|34914|
|     TR_V_pseudogene|   33|
|           vault_RNA|    4|
|               snRNA| 1901|
|transcribed_proce...| 1159|
|           IG_C_gene|   14|
|           IG_J_gene|   18|
|  unitary_pseudogene|   79|
|           TR_V_gene|  112|
|      protein_coding|20128|
|processed_pseudogene| 9488|
|               miRNA| 1879|
|translated_proces...|    2|
|           TR_J_gene|   79|
|transcribed_unpro...| 1600|
|              snoRNA|  942|
|     rRNA_pseudogene|  497|
+--------------------+-----+
only showing top 20 rows



In [ ]:
target.count()

78766

In [ ]:
target_pc=target.filter(f.col("biotype")=="protein_coding").cache()
target_pc.count()

20128

In [ ]:
id_target_pc=target_pc.select("id").distinct().withColumnRenamed("id","targetId").cache()
id_target_pc.count()

20128

In [ ]:
target_constr=target_constr.join(id_target_pc, on="targetId", how="inner").withColumnRenamed("targetId","geneId").cache()
target_constr.count()

20128

In [ ]:
combined_result = target_constr.join(
    aggregated_eqtls,
    on="geneId",
    how="left"
)
combined_result.count()

20128

In [ ]:
combined_result.show(2)

+---------------+------------+----------+--------------+---------+---------+----------------------+-------------------+----------------------------+----------------------------------+------------------+------+-------------------+----------------------------+---------------------+-----------------+------------------+------------------------+
|         geneId|isInMembrane|isSecreted|hasSafetyEvent|hasPocket|hasLigand|hasSmallMoleculeBinder|  geneticConstraint|paralogMaxIdentityPercentage|mouseOrthologMaxIdentityPercentage|isCancerDriverGene|hasTEP|       mouseKOScore|hasHighQualityChemicalProbes|maxClinicalTrialPhase|tissueSpecificity|tissueDistribution|distinct_biosample_count|
+---------------+------------+----------+--------------+---------+---------+----------------------+-------------------+----------------------------+----------------------------------+------------------+------+-------------------+----------------------------+---------------------+-----------------+----------------

In [ ]:
combined_result.filter(f.col("distinct_biosample_count")>0).count()


17392

## L2G

In [ ]:
l2g=session.spark.read.parquet(path_to_release_folder+"output/l2g_prediction")

In [ ]:
qcs=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/qualified_CSs_with_oncology")

In [ ]:
l2g=l2g.join(qcs, on="studyLocusId", how="inner").cache()
l2g.count()

92165

In [ ]:
l2g=l2g.filter(f.col("score")>=0.5).select("geneId").distinct().cache()
l2g.count()

5995

In [ ]:
l2g=l2g.withColumn("l2g", f.lit(1)).cache()
l2g.count()

5995

In [ ]:
combined_result = combined_result.join(
    l2g,
    on="geneId",
    how="left"
)
combined_result.count()

20128

In [ ]:
combined_result.filter(f.col("l2g")>0).count()

5995

In [ ]:
combined_result.show(2)

+---------------+------------+----------+--------------+---------+---------+----------------------+-------------------+----------------------------+----------------------------------+------------------+------+-------------------+----------------------------+---------------------+-----------------+------------------+------------------------+----+
|         geneId|isInMembrane|isSecreted|hasSafetyEvent|hasPocket|hasLigand|hasSmallMoleculeBinder|  geneticConstraint|paralogMaxIdentityPercentage|mouseOrthologMaxIdentityPercentage|isCancerDriverGene|hasTEP|       mouseKOScore|hasHighQualityChemicalProbes|maxClinicalTrialPhase|tissueSpecificity|tissueDistribution|distinct_biosample_count| l2g|
+---------------+------------+----------+--------------+---------+---------+----------------------+-------------------+----------------------------+----------------------------------+------------------+------+-------------------+----------------------------+---------------------+-----------------+------

In [60]:
combined_result.write.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/protein_coding_eqtls_l2g",mode="overwrite")

In [2]:
combined_result=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/protein_coding_eqtls_l2g")

In [3]:
combined_result.show(1)

+---------------+------------+----------+--------------+---------+---------+----------------------+------------------+----------------------------+----------------------------------+------------------+------+------------+----------------------------+---------------------+-----------------+------------------+------------------------+----+
|         geneId|isInMembrane|isSecreted|hasSafetyEvent|hasPocket|hasLigand|hasSmallMoleculeBinder| geneticConstraint|paralogMaxIdentityPercentage|mouseOrthologMaxIdentityPercentage|isCancerDriverGene|hasTEP|mouseKOScore|hasHighQualityChemicalProbes|maxClinicalTrialPhase|tissueSpecificity|tissueDistribution|distinct_biosample_count| l2g|
+---------------+------------+----------+--------------+---------+---------+----------------------+------------------+----------------------------+----------------------------------+------------------+------+------------+----------------------------+---------------------+-----------------+------------------+-----------

In [4]:
combined_result.repartition(1).write.csv("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/protein_coding_eqtls_l2g.csv",mode="overwrite",header=True)

25/04/26 00:28:21 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 969216 ms exceeds timeout 120000 ms
25/04/26 00:28:21 WARN SparkContext: Killing executors is not supported by current scheduler.
25/04/26 00:28:23 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

# VEP, eQTLs, and other fm-based evidnece

In [3]:
combined_result=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/protein_coding_eqtls_l2g")

In [4]:
fm=session.spark.read.parquet(path_to_release_folder+"intermediate/l2g_feature_matrix/")
fm=fm.filter(f.col("isProteinCoding")==1).cache()
fm.count()

25/04/29 14:53:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


7734143

In [5]:
qsl=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/qualified_CSs_with_oncology")

In [6]:
fm=fm.join(qsl, on="studyLocusId", how="inner").cache()
fm.count()

656596

In [7]:
clpp_thr=0.01
coloc_thr=0.8

combined_df = fm.withColumn(
    "eQTL_coloc",
    f.when(
    (f.col("eQtlColocClppMaximum")>=clpp_thr) | 
    (f.col("eQtlColocH4Maximum")>=coloc_thr), 1).otherwise(0)
)
combined_df = combined_df.withColumn(
    "pQTL_coloc",
    f.when(
    (f.col("pQtlColocClppMaximum")>=clpp_thr) | 
    (f.col("pQtlColocH4Maximum")>=coloc_thr), 1).otherwise(0)
)
combined_df = combined_df.withColumn(
    "sQTL_coloc",
    f.when(
    (f.col("sQtlColocClppMaximum")>=clpp_thr) | 
    (f.col("sQtlColocH4Maximum")>=coloc_thr), 1).otherwise(0)
)
combined_df = combined_df.withColumn(
    "VEP",
    f.when((f.col("vepMaximum")>=0.66), 1).otherwise(0)
)
combined_df = combined_df.withColumn(
    "distance",
    f.when((f.col("distanceSentinelFootprintNeighbourhood")==1) |
    (f.col("distanceSentinelTssNeighbourhood")==1), 1).otherwise(0)
).cache()

combined_df.count()

656596

In [8]:
combined_df.show(1)

+--------------------+---------------+---------------------+---------------------+----------------------------------+-------------------------+--------------------------------------+-------------------+--------------------------------+---------------+----------------------------+--------------------+---------------------------------+------------------+-------------------------------+--------------+---------------+--------------------+---------------------------------+------------------+-------------------------------+---------------------+--------------------+---------------------------------+------------------+-------------------------------+----------+-----------------------+-------+--------------------+----------+----------+----------+---+--------+
|        studyLocusId|         geneId|credibleSetConfidence|distanceFootprintMean|distanceFootprintMeanNeighbourhood|distanceSentinelFootprint|distanceSentinelFootprintNeighbourhood|distanceSentinelTss|distanceSentinelTssNeighbourhood|dis

In [9]:
VEP_g=combined_df.filter(f.col("VEP")==1).select("geneId","VEP").distinct().cache()
VEP_g.count()

2605

In [10]:
combined_result = combined_result.join(
    VEP_g,
    on="geneId",
    how="left"
).cache()
combined_result.count()

20128

In [11]:
eQTL_coloc=combined_df.filter(f.col("eQTL_coloc")==1).select("geneId","eQTL_coloc").distinct().cache()
eQTL_coloc.count()

6421

In [12]:
combined_result = combined_result.join(
    eQTL_coloc,
    on="geneId",
    how="left"
).cache()
combined_result.count()

20128

In [13]:
pQTL_coloc=combined_df.filter(f.col("pQTL_coloc")==1).select("geneId","pQTL_coloc").distinct().cache()
pQTL_coloc.count()

528

In [14]:
combined_result = combined_result.join(
    pQTL_coloc,
    on="geneId",
    how="left"
).cache()
combined_result.count()

20128

In [15]:
sQTL_coloc=combined_df.filter(f.col("sQTL_coloc")==1).select("geneId","sQTL_coloc").distinct().cache()
sQTL_coloc.count()

4023

In [16]:
combined_result = combined_result.join(
    sQTL_coloc,
    on="geneId",
    how="left"
).cache()
combined_result.count()

20128

In [17]:
distance=combined_df.filter(f.col("distance")==1).select("geneId","distance").distinct().cache()
distance.count()

8524

In [18]:
combined_result = combined_result.join(
    distance,
    on="geneId",
    how="left"
).cache()
combined_result.count()

20128

In [19]:
combined_result.show(2)

+---------------+------------+----------+--------------+---------+---------+----------------------+-------------------+----------------------------+----------------------------------+------------------+------+------------+----------------------------+---------------------+-----------------+------------------+------------------------+----+----+----------+----------+----------+--------+
|         geneId|isInMembrane|isSecreted|hasSafetyEvent|hasPocket|hasLigand|hasSmallMoleculeBinder|  geneticConstraint|paralogMaxIdentityPercentage|mouseOrthologMaxIdentityPercentage|isCancerDriverGene|hasTEP|mouseKOScore|hasHighQualityChemicalProbes|maxClinicalTrialPhase|tissueSpecificity|tissueDistribution|distinct_biosample_count| l2g| VEP|eQTL_coloc|pQTL_coloc|sQTL_coloc|distance|
+---------------+------------+----------+--------------+---------+---------+----------------------+-------------------+----------------------------+----------------------------------+------------------+------+------------+--

In [20]:
combined_result.write.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/protein_coding_eqtls_l2g_fm_based",mode="overwrite")

# Other evidence

In [21]:
evidence=session.spark.read.parquet(path_to_release_folder+"/output/evidence/")

In [22]:
evidence = evidence.filter(f.col("score")>=0.5).filter(f.col("datasourceId").isin(["eva", "uniprot_variants", "gene2phenotype", "genomics_england", "clingen"])).cache()
evidence.count()

539783

In [23]:
evidence.printSchema()

root
 |-- datasourceId: string (nullable = true)
 |-- targetId: string (nullable = true)
 |-- alleleOrigins: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- allelicRequirements: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ancestry: string (nullable = true)
 |-- ancestryId: string (nullable = true)
 |-- beta: double (nullable = true)
 |-- betaConfidenceIntervalLower: double (nullable = true)
 |-- betaConfidenceIntervalUpper: double (nullable = true)
 |-- biologicalModelAllelicComposition: string (nullable = true)
 |-- biologicalModelGeneticBackground: string (nullable = true)
 |-- biologicalModelId: string (nullable = true)
 |-- biomarkerName: string (nullable = true)
 |-- biomarkers: struct (nullable = true)
 |    |-- geneExpression: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |-- genetic

In [24]:
other_evid=evidence.select("targetId").distinct()
other_evid=other_evid.withColumnRenamed("targetId","geneId")
other_evid=other_evid.withColumn("rare_variant_evidence", f.lit(1)).cache()
other_evid.count()

5892

In [25]:
combined_result = combined_result.join(
    other_evid,
    on="geneId",
    how="left"
).cache()
combined_result.count()

20128

In [26]:
combined_result.show(1)

+---------------+------------+----------+--------------+---------+---------+----------------------+------------------+----------------------------+----------------------------------+------------------+------+------------+----------------------------+---------------------+-----------------+------------------+------------------------+----+----+----------+----------+----------+--------+---------------------+
|         geneId|isInMembrane|isSecreted|hasSafetyEvent|hasPocket|hasLigand|hasSmallMoleculeBinder| geneticConstraint|paralogMaxIdentityPercentage|mouseOrthologMaxIdentityPercentage|isCancerDriverGene|hasTEP|mouseKOScore|hasHighQualityChemicalProbes|maxClinicalTrialPhase|tissueSpecificity|tissueDistribution|distinct_biosample_count| l2g| VEP|eQTL_coloc|pQTL_coloc|sQTL_coloc|distance|rare_variant_evidence|
+---------------+------------+----------+--------------+---------+---------+----------------------+------------------+----------------------------+----------------------------------+

In [27]:
combined_result.write.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/protein_coding_eqtls_l2g_fm_based_rare_varint",mode="overwrite")

In [ ]:
#combined_result.repartition(1).write.csv("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/protein_coding_eqtls_l2g_fm_based_rare_varint.csv",mode="overwrite",header=True)

25/04/30 23:16:54 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1948484 ms exceeds timeout 120000 ms
25/04/30 23:16:54 WARN SparkContext: Killing executors is not supported by current scheduler.
25/04/30 23:16:57 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$

# Danile's table

In [2]:
df=session.spark.read.parquet("gs://genetics-portal-dev-analysis/dc16/output/pleiotropy_genes_therapeutic_areas")
df.count()

6999

In [3]:
df.show(1)

25/05/15 17:21:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------------+----------+-----------+---------+----------+--------------+---------+------+----------+-----------+-------------+--------+--------------+-----------+---------+---------+---------------+---------+-----+------------+--------------+--------------+----------+---------+---------+-------------+-------------+-------------+----------+-----------------+------------------+
|         geneId|uniqueEFOs|Haematology|Metabolic|Congenital|Signs/symptoms|Neurology|Immune|Psychiatry|Dermatology|Ophthalmology|Oncology|Cardiovascular|Respiratory|Digestive|Endocrine|Musculoskeletal|Infection|Other|totalStudies|approvedSymbol|       biotype|chromosome|    start|      end|lofConstraint|misConstraint|synConstraint|pleiotropy|tissueSpecificity|tissueDistribution|
+---------------+----------+-----------+---------+----------+--------------+---------+------+----------+-----------+-------------+--------+--------------+-----------+---------+---------+---------------+---------+-----+------------+---

In [31]:
df=df.drop("tissueSpecificity","tissueDistribution")

In [32]:
combined_result = combined_result.join(
    df,
    on="geneId",
    how="left"
).cache()
combined_result.count()

20128

In [33]:
combined_result.write.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/protein_coding_eqtls_l2g_fm_based_rare_varint_pleiotropy",mode="overwrite")

In [34]:
combined_result.repartition(1).write.csv("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/protein_coding_eqtls_l2g_fm_based_rare_varint_pleiotropy.csv",mode="overwrite",header=True)

# Daniel's table

In [12]:
x=session.spark.read.parquet("gs://genetics-portal-dev-analysis/dc16/output/qsl_l2g_pleiotropy")

In [13]:
x.printSchema()

root
 |-- geneId: string (nullable = true)
 |-- studyLocusId: string (nullable = true)
 |-- variantId: string (nullable = true)
 |-- studyId: string (nullable = true)
 |-- beta: double (nullable = true)
 |-- zScore: double (nullable = true)
 |-- pValueMantissa: float (nullable = true)
 |-- pValueExponent: integer (nullable = true)
 |-- standardError: double (nullable = true)
 |-- finemappingMethod: string (nullable = true)
 |-- studyType: string (nullable = true)
 |-- credibleSetSize: integer (nullable = true)
 |-- posteriorProbability: double (nullable = true)
 |-- nSamples: integer (nullable = true)
 |-- nControls: integer (nullable = true)
 |-- nCases: integer (nullable = true)
 |-- majorPopulation: struct (nullable = true)
 |    |-- ldPopulation: string (nullable = true)
 |    |-- relativeSampleSize: double (nullable = true)
 |-- allelefrequencies: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- populationName: string (nullable = true)
 |    |

In [ ]:
#x.write.parquet("./data/qsl_l2g_pleiotropy")

In [ ]:
#gns=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/protein_coding_eqtls_l2g_fm_based_rare_varint_pleiotropy")

In [ ]:
#gns.count()

20128

In [ ]:
#gns.printSchema()

root
 |-- geneId: string (nullable = true)
 |-- isInMembrane: integer (nullable = true)
 |-- isSecreted: integer (nullable = true)
 |-- hasSafetyEvent: integer (nullable = true)
 |-- hasPocket: integer (nullable = true)
 |-- hasLigand: integer (nullable = true)
 |-- hasSmallMoleculeBinder: integer (nullable = true)
 |-- geneticConstraint: double (nullable = true)
 |-- paralogMaxIdentityPercentage: double (nullable = true)
 |-- mouseOrthologMaxIdentityPercentage: double (nullable = true)
 |-- isCancerDriverGene: integer (nullable = true)
 |-- hasTEP: integer (nullable = true)
 |-- mouseKOScore: double (nullable = true)
 |-- hasHighQualityChemicalProbes: integer (nullable = true)
 |-- maxClinicalTrialPhase: double (nullable = true)
 |-- tissueSpecificity: double (nullable = true)
 |-- tissueDistribution: double (nullable = true)
 |-- distinct_biosample_count: long (nullable = true)
 |-- l2g: integer (nullable = true)
 |-- VEP: integer (nullable = true)
 |-- eQTL_coloc: integer (nullable 

In [4]:
fm=session.spark.read.parquet(path_to_release_folder+"intermediate/l2g_feature_matrix/")
fm=fm.filter(f.col("isProteinCoding")==1).cache()
fm.count()

7734143

In [6]:
combined_df=fm
clpp_thr=0.01
coloc_thr=0.8

combined_df = combined_df.withColumn(
    "eQTL_coloc",
    f.when(
    (f.col("eQtlColocClppMaximum")>=clpp_thr) | 
    (f.col("eQtlColocH4Maximum")>=coloc_thr), 1).otherwise(0)
)
combined_df = combined_df.withColumn(
    "pQTL_coloc",
    f.when(
    (f.col("pQtlColocClppMaximum")>=clpp_thr) | 
    (f.col("pQtlColocH4Maximum")>=coloc_thr), 1).otherwise(0)
)
combined_df = combined_df.withColumn(
    "sQTL_coloc",
    f.when(
    (f.col("sQtlColocClppMaximum")>=clpp_thr) | 
    (f.col("sQtlColocH4Maximum")>=coloc_thr), 1).otherwise(0)
)
combined_df = combined_df.withColumn(
    "VEP",
    f.when((f.col("vepMaximum")>=0.66), 1).otherwise(0)
)


combined_df = combined_df.select("VEP","eQTL_coloc","pQTL_coloc","sQTL_coloc","studyLocusId","geneId").cache()


combined_df.count()

7734143

In [7]:
combined_df.show(1)

+---+----------+----------+----------+--------------------+---------------+
|VEP|eQTL_coloc|pQTL_coloc|sQTL_coloc|        studyLocusId|         geneId|
+---+----------+----------+----------+--------------------+---------------+
|  0|         0|         0|         0|0051069c8d2eb2f07...|ENSG00000282757|
+---+----------+----------+----------+--------------------+---------------+
only showing top 1 row



In [8]:
ta=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/gwas_study_index_with_theraputic_areas").cache()

In [9]:
ta.printSchema()

root
 |-- studyId: string (nullable = true)
 |-- projectId: string (nullable = true)
 |-- studyType: string (nullable = true)
 |-- traitFromSource: string (nullable = true)
 |-- traitFromSourceMappedIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- diseaseIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- geneId: string (nullable = true)
 |-- biosampleFromSourceId: string (nullable = true)
 |-- biosampleId: string (nullable = true)
 |-- pubmedId: string (nullable = true)
 |-- publicationTitle: string (nullable = true)
 |-- publicationFirstAuthor: string (nullable = true)
 |-- publicationDate: string (nullable = true)
 |-- publicationJournal: string (nullable = true)
 |-- backgroundTraitFromSourceMappedIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- backgroundDiseaseIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- initialSampleSize: string (nullable = true)
 

In [10]:
ta=ta.select("studyId","Haematology","Metabolic","Congenital","Signs/symptoms","Neurology","Immune","Psychiatry","Dermatology",
             "Ophthalmology","Cardiovascular","Oncology","Respiratory","Digestive","Endocrine","Musculoskeletal","Infection","Measurement","Other").cache()

In [14]:
x.count()

39053

In [15]:
x=x.join(ta, on="studyId", how="inner")
x=x.join(combined_df, on=["studyLocusId","geneId"], how="inner").cache()
x.count()

39053

In [ ]:
x.write.parquet("./data/qsl_l2g_pleiotropy_ta_reasons_of_assoc")

25/05/16 07:04:52 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 934940 ms exceeds timeout 120000 ms
25/05/16 07:04:52 WARN SparkContext: Killing executors is not supported by current scheduler.
25/05/16 07:05:00 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$